## Setup

In [17]:
from pathlib import Path
import numpy as np
import pandas as pd
from Bio import SeqIO
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

from collections import Counter
from itertools import product

In [2]:
data_dir = Path("../data")
raw_dir = data_dir / "raw"

for item in raw_dir.iterdir(): 
    print(item)

..\data\raw\.gitkeep
..\data\raw\cytb.txt
..\data\raw\mc1r.txt
..\data\raw\morfometricos.xlsx


In [3]:
mc1r_path = raw_dir / "mc1r.txt"
cytb_path = raw_dir / "cytb.txt"

assert mc1r_path.is_file() and cytb_path.is_file()

## Carga de datos

In [4]:
def fasta_to_df(filepath) -> pd.DataFrame:
    records = []

    for record in SeqIO.parse(filepath, "fasta"): 
        records.append({
            'id': record.id, 
            'sequence': str(record.seq), 
        })

    df = pd.DataFrame(records)
    return df

In [5]:
mc1r_df = fasta_to_df(mc1r_path)
mc1r_df.head()

,id,sequence
0,KR026455,CGGAGTTGCAAATGACAAGGATGAGGTAGAGGCTGAAGTAGCTGAA...
1,KR026456,CGGAGTTGCAAATGACAAGGATGAGGTAGAGGCTGAAGTAGCTGAA...
2,KR026457,CGGAGTTGCAAATGACAAGGATGAGGTAGAGGCTGAAGTAGCTGAA...
3,KR026458,CGGAGTTGCAAATGACAAGGATGAGGTAGAGGCTGAAGTAGCTGAA...
4,KR026459,CGGAGTTGCAAATGACAAGGATGAGGTAGAGGCTGAAGTAGCTGAA...


In [6]:
mc1r_df.shape

(43, 2)

In [8]:
cytb_df = fasta_to_df(cytb_path)
cytb_df.head()

,id,sequence
0,KR026343,GCCTAATTATCCAAATCCTAACAGGACTATTTCTAGCCATACACTA...
1,KR026344,GCCTAATTATCCAAATCCTAACAGGACTATTTCTAGCCATACACTA...
2,KR026345,GCCTAATTATCCAAATCCTAACAGGACTATTTCTAGCCATACACTA...
3,KR026346,GCCTAATTATCCAAATCCTAACAGGACTATTTCTAGCCATACACTA...
4,KR026347,GCCTAATTATCCAAATCCTAACAGGACTATTTCTAGCCATACACTA...


In [9]:
cytb_df.shape

(36, 2)

- El indicador genético `mc1r` posee 43 registros, mientras que el indicador `cytb` cuenta con 36 registros. 

## Validación y limpieza

La longitud de las cadenas: 

In [10]:
mc1r_df['length'] = mc1r_df['sequence'].str.len()
cytb_df['length'] = cytb_df['sequence'].str.len()

In [11]:
mc1r_df['length'].value_counts()

length
815    43
Name: count, dtype: int64

- Todos los registros considerados para el indicador `mc1r` poseen cadenas con 815 nucleótidos

In [12]:
cytb_df['length'].value_counts()

length
1027    35
1081     1
Name: count, dtype: int64

- Uno de los 36 registros posee cadena con más nucleótidos que el resto. 

Verificamos el registro anormal: 

In [13]:
cytb_df.query("length != 1027")

,id,sequence,length
35,Holbrookia,CTTTGGTTCACTACTAGGAATTTGCCTGATTATTCAAATCTTAACA...,1081


Para evitar la introducción de ruido, eliminaremos este registro. 

In [62]:
cytb_df = cytb_df.drop(axis=0, index=35)

In [63]:
assert cytb_df['length'].nunique() == 1

Este registro también se encuentra en el dataset del indicador `mc1r`. Lo eliminaremos para ser congruentes. 

In [64]:
mc1r_df.iloc[-1]

id                                                 Holbrookia
sequence    CGGAGTTGCAAATGACAAGGATGAGGTAGAGGCTGAAGTAGCTGAA...
length                                                    815
Name: 42, dtype: object

In [65]:
mc1r_df.query("id == 'Holbrookia'")

,id,sequence,length
42,Holbrookia,CGGAGTTGCAAATGACAAGGATGAGGTAGAGGCTGAAGTAGCTGAA...,815


In [66]:
mc1r_df = mc1r_df.drop(axis=0, index=42)

Los nucleótidos en cada registro deben ser el mismo: 

In [14]:
VALID_BASES = set('ACGT')

def validate_sequences(df):
    invalid = []

    for idx, seq in enumerate(df['sequence']):
        chars = set(seq)

        if not chars.issubset(VALID_BASES):
            invalid.append(idx)

    return invalid

In [15]:
assert not validate_sequences(mc1r_df), 'Nucleótido invalido'

In [16]:
assert not validate_sequences(cytb_df), 'Nucleótido invalido'

## Matriz genética

Para hacer inferencia, es necesario reprecentar los registros como una matriz genética. La idea es convertir las secuencias de nucleótidos en variables conparables.

In [67]:
def sequences_to_matrix(df):
    sequences = df['sequence'].tolist()

    matrix = pd.DataFrame(
        [list(seq) for seq in sequences]
    )

    matrix.columns = [i for i in range(matrix.shape[1])]

    matrix.insert(0, 'id', df['id'].values)

    return matrix

In [68]:
mc1r_matrix = sequences_to_matrix(mc1r_df)
mc1r_matrix.head()

,id,0,1,2,3,4,5,6,7,8,...,805,806,807,808,809,810,811,812,813,814
0,KR026455,C,G,G,A,G,T,T,G,C,...,G,T,C,A,C,T,G,G,C,A
1,KR026456,C,G,G,A,G,T,T,G,C,...,G,T,C,G,C,T,G,G,C,A
2,KR026457,C,G,G,A,G,T,T,G,C,...,G,T,C,G,C,T,G,G,C,A
3,KR026458,C,G,G,A,G,T,T,G,C,...,G,T,C,G,C,T,G,G,C,A
4,KR026459,C,G,G,A,G,T,T,G,C,...,G,T,C,G,C,T,G,G,C,A


In [69]:
cytb_matrix = sequences_to_matrix(cytb_df)
cytb_matrix.head()

,id,0,1,2,3,4,5,6,7,8,...,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026
0,KR026343,G,C,C,T,A,A,T,T,A,...,G,A,T,A,A,A,A,T,T,T
1,KR026344,G,C,C,T,A,A,T,T,A,...,G,A,T,A,A,A,A,T,T,T
2,KR026345,G,C,C,T,A,A,T,T,A,...,G,A,T,A,A,A,A,T,T,T
3,KR026346,G,C,C,T,A,A,T,T,A,...,G,A,T,A,A,A,A,T,T,T
4,KR026347,G,C,C,T,A,A,T,T,A,...,G,A,T,A,A,A,A,T,T,T


## Codificación numérica

### Labeled encoding

Para trabajar con modelos matemáticos, es necesario reprecentar la información genética de manera numérica. Haremos esto mapeando el identificador de nucleótido a un número entero: 

    > {A: 1, C: 1, G: 2, T: 3}

In [70]:
ENCODING = {
    'A': 0,
    'C': 1,
    'G': 2,
    'T': 3
}

def encode_matrix(df):
    encoded = df.copy()

    feature_cols = encoded.columns[1:]

    for col in feature_cols:
        encoded[col] = encoded[col].map(ENCODING)

    return encoded

In [71]:
mc1r_encoded = encode_matrix(mc1r_matrix)
mc1r_encoded.head()

,id,0,1,2,3,4,5,6,7,8,...,805,806,807,808,809,810,811,812,813,814
0,KR026455,1,2,2,0,2,3,3,2,1,...,2,3,1,0,1,3,2,2,1,0
1,KR026456,1,2,2,0,2,3,3,2,1,...,2,3,1,2,1,3,2,2,1,0
2,KR026457,1,2,2,0,2,3,3,2,1,...,2,3,1,2,1,3,2,2,1,0
3,KR026458,1,2,2,0,2,3,3,2,1,...,2,3,1,2,1,3,2,2,1,0
4,KR026459,1,2,2,0,2,3,3,2,1,...,2,3,1,2,1,3,2,2,1,0


In [72]:
cytb_encoded = encode_matrix(cytb_matrix)
cytb_encoded.head()

,id,0,1,2,3,4,5,6,7,8,...,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026
0,KR026343,2,1,1,3,0,0,3,3,0,...,2,0,3,0,0,0,0,3,3,3
1,KR026344,2,1,1,3,0,0,3,3,0,...,2,0,3,0,0,0,0,3,3,3
2,KR026345,2,1,1,3,0,0,3,3,0,...,2,0,3,0,0,0,0,3,3,3
3,KR026346,2,1,1,3,0,0,3,3,0,...,2,0,3,0,0,0,0,3,3,3
4,KR026347,2,1,1,3,0,0,3,3,0,...,2,0,3,0,0,0,0,3,3,3


### One-hot encoding

Para evitar encodings arbitrarios que puedan afectar artificialmente las distancias entre secuencias tambien reqalizamos un one-hot encoding, concatenando los valores para evitar crear un tensor.

In [106]:
def oneHotEncoding(df):
    one_hot_list = []
    for col in df.columns:
        dummies = pd.get_dummies(df[col], prefix=col, prefix_sep='_')
        one_hot_list.append(dummies)

    return pd.concat(one_hot_list, axis=1).fillna(0).astype(int)

In [110]:
mc1r_oneHot = oneHotEncoding(mc1r_matrix.iloc[:,1:])
mc1r_oneHot.head()

,0_C,1_G,2_G,3_A,4_G,5_T,6_T,7_G,8_C,9_A,...,806_T,807_C,808_A,808_G,809_C,810_T,811_G,812_G,813_C,814_A
0,1,1,1,1,1,1,1,1,1,1,...,1,1,1,0,1,1,1,1,1,1
1,1,1,1,1,1,1,1,1,1,1,...,1,1,0,1,1,1,1,1,1,1
2,1,1,1,1,1,1,1,1,1,1,...,1,1,0,1,1,1,1,1,1,1
3,1,1,1,1,1,1,1,1,1,1,...,1,1,0,1,1,1,1,1,1,1
4,1,1,1,1,1,1,1,1,1,1,...,1,1,0,1,1,1,1,1,1,1


In [112]:
cytb_oneHot = oneHotEncoding(cytb_matrix.iloc[:,1:])
cytb_oneHot.head()

,0_G,1_C,2_C,3_T,4_A,5_A,6_T,7_C,7_T,8_A,...,1017_G,1018_A,1019_T,1020_A,1021_A,1022_A,1023_A,1024_T,1025_T,1026_T
0,1,1,1,1,1,1,1,0,1,1,...,1,1,1,1,1,1,1,1,1,1
1,1,1,1,1,1,1,1,0,1,1,...,1,1,1,1,1,1,1,1,1,1
2,1,1,1,1,1,1,1,0,1,1,...,1,1,1,1,1,1,1,1,1,1
3,1,1,1,1,1,1,1,0,1,1,...,1,1,1,1,1,1,1,1,1,1
4,1,1,1,1,1,1,1,0,1,1,...,1,1,1,1,1,1,1,1,1,1


la unica desventaja es que no son comparabres entre dataFrames

### k-meros

Distancia genetica que mide la frecuencia de las subsecuencias de longitud $k$ de una secuencia. Bastante similar a los n-gramas en Procesamiento de Lenguaje Natural.

En este caso consideramos pertinente usar $k=4$, debido a que los sesgos hacia tetranucleótidos de organismos filogenéticamente más relacionados son más parecidos entre sí que entre organismos menos emparentados.

Entonces convertimos una secuencia (de longitud arbitraria, no necesitan ser iguales) en un vector de numeros flotantes de $4^4 = 256$ elementos. Y asi la distancia euclidiana entre ellos nos da la distancia genetica entre secuencias.

In [55]:
def kmer_freq(seq, k=4):
    """
    Convierte una cadena de texto (formato FASTA) en un vector de las frecuencias de los k-meros
    """
    seq = seq[0].upper().replace('\n', '').replace(' ', '')
    kmers = [seq[i:i+k] for i in range(len(seq) - k + 1)]
    counts = Counter(kmers)
    total = len(kmers)+1 # para evitar division por 0
    # Regresa las frecuencias de todos los posibles k-meros en orden alfabetico
    all_kmers = [''.join(x) for x in product('ACGT', repeat=k)]
    return np.array([counts.get(kmer, 0) / total for kmer in all_kmers])

In [ ]:
cytb_df["sequence"][35]

1081

In [58]:
cytb_mers = cytb_df[["sequence"]].apply(kmer_freq, axis=1, result_type="expand")
cytb_mers.head()

C:\Users\DELL\AppData\Local\Temp\ipykernel_21024\1682717147.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  seq = seq[0].upper().replace('\n', '').replace(' ', '')


,0,1,2,3,4,5,6,7,8,9,...,246,247,248,249,250,251,252,253,254,255
0,0.003902,0.010732,0.002927,0.007805,0.014634,0.007805,0.003902,0.003902,0.001951,0.000976,...,0.000976,0.00878,0.001951,0.003902,0.006829,0.000976,0.009756,0.009756,0.002927,0.01561
1,0.003902,0.010732,0.002927,0.007805,0.014634,0.007805,0.003902,0.003902,0.001951,0.000976,...,0.000976,0.00878,0.001951,0.003902,0.006829,0.000976,0.008780,0.009756,0.002927,0.01561
2,0.003902,0.010732,0.002927,0.007805,0.014634,0.007805,0.003902,0.003902,0.001951,0.000976,...,0.000976,0.00878,0.001951,0.003902,0.006829,0.000976,0.009756,0.009756,0.002927,0.01561
3,0.003902,0.010732,0.002927,0.007805,0.014634,0.007805,0.003902,0.003902,0.001951,0.000976,...,0.000976,0.00878,0.001951,0.003902,0.006829,0.000976,0.009756,0.009756,0.002927,0.01561
4,0.003902,0.010732,0.002927,0.007805,0.014634,0.007805,0.003902,0.003902,0.001951,0.000976,...,0.000976,0.00878,0.001951,0.003902,0.006829,0.000976,0.009756,0.009756,0.002927,0.01561


In [59]:
mc1r_mers = mc1r_df[["sequence"]].apply(kmer_freq, axis=1, result_type="expand")
mc1r_mers.head()

C:\Users\DELL\AppData\Local\Temp\ipykernel_21024\1682717147.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  seq = seq[0].upper().replace('\n', '').replace(' ', '')


,0,1,2,3,4,5,6,7,8,9,...,246,247,248,249,250,251,252,253,254,255
0,0.00123,0.00123,0.00246,0.00246,0.00492,0.0,0.0,0.00246,0.0123,0.0,...,0.00123,0.00369,0.00246,0.00246,0.00492,0.00246,0.0,0.00123,0.0,0.0
1,0.00123,0.00123,0.00246,0.00246,0.00492,0.0,0.0,0.00246,0.0123,0.0,...,0.00123,0.00369,0.00246,0.00246,0.00369,0.00246,0.0,0.00123,0.0,0.0
2,0.00123,0.00123,0.00246,0.00246,0.00492,0.0,0.0,0.00246,0.0123,0.0,...,0.00123,0.00369,0.00246,0.00246,0.00369,0.00246,0.0,0.00123,0.0,0.0
3,0.00123,0.00123,0.00246,0.00246,0.00492,0.0,0.0,0.00246,0.0123,0.0,...,0.00123,0.00369,0.00246,0.00246,0.00492,0.00246,0.0,0.00123,0.0,0.0
4,0.00123,0.00123,0.00246,0.00246,0.00615,0.0,0.0,0.00246,0.0123,0.0,...,0.00123,0.00369,0.00246,0.00246,0.00369,0.00246,0.0,0.00123,0.0,0.0


## Exportación

Guardamos ambos datasets en el directorio `data/processed`: 

In [73]:
cytb_output_path = Path("../data/processed/cytb_encoded.csv")
cytb_encoded.to_csv(
    cytb_output_path,
    index=False
)

In [74]:
mc1r_output_path = Path("../data/processed/mc1r_encoded.csv")
mc1r_encoded.to_csv(
    mc1r_output_path,
    index=False
)

In [113]:
cytb_output_path = Path("../data/processed/cytb_one_hot_encoded.csv")
cytb_oneHot.to_csv(
    cytb_output_path,
    index=False
)

In [114]:
mc1r_output_path = Path("../data/processed/mc1r_one_hot_encoded.csv")
mc1r_oneHot.to_csv(
    mc1r_output_path,
    index=False
)

In [60]:
cytb_output_path = Path("../data/processed/cytb_4mers.csv")
cytb_mers.to_csv(
    cytb_output_path,
    index=False
)

In [61]:
mc1r_output_path = Path("../data/processed/mc1r_4mers.csv")
mc1r_mers.to_csv(
    mc1r_output_path,
    index=False
)